# Load deps

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from sqlitesearch import TextSearchIndex

from src.ingest import load_faq_data, build_index, build_sqlite_index
from src.rag_helper import RAGBase

# Load documents

In [ ]:
documents = load_faq_data()

# Index documents

In [ ]:
index = build_index(documents)

In [ ]:
sqlite_index = build_sqlite_index(
    [doc for doc in documents if doc["course"] == "llm-zoomcamp"],
    db_path="KB/llmzc_faq.db"
)

# Create LLM client

In [ ]:
openai_client = OpenAI()

# Load prompt components

In [ ]:
with open("templates/instructions.txt", "r") as file:
    instructions = file.read().strip()
with open("templates/prompt_template.txt", "r") as file:
    prompt_template = file.read().strip()
# print(instructions)
# print(prompt_template)

# Create RAG instance

In [ ]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
    prompt_template=prompt_template
)

# Run the rag pipeline

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = assistant.rag(question)
print(answer.output_text)

# Run the rag with sqlitesearch

In [ ]:
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="KB/llmzc_faq.db"
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    instructions=instructions,
    prompt_template=prompt_template
)

answer = assistant.rag(question)
print(answer.output_text)

In [ ]:
sqlite_index.close()

# Agentic RAG

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
agent_tools.get_tools()

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

In [ ]:
# print(result.cost)
# print(result.all_messages)

### Continuing the conversation

In [ ]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

### Interactive chat

In [ ]:
runner.run()
# Type "stop" to exit.